---
## Key Takeaways

1. **Docker** = reproducible environments — eliminates "works on my machine"
2. **FastAPI** = type-safe, self-documenting API — Pydantic validates inputs automatically
3. **Load model at startup** (`@app.on_event("startup")`) — not on every request
4. **/health endpoint** = required for load balancers and Kubernetes
5. **Batch endpoints** — more efficient than N individual calls for ML workloads
6. **/docs** — free interactive documentation from FastAPI

---
## Exercises

### Exercise 1: Add Validation
Add a custom validator that rejects monthly_charges > 10 * tenure_months (clearly invalid data).

### Exercise 2: Add Caching
Add a simple in-memory cache using a dict: if the exact same input has been seen in the last 5 minutes, return the cached result without calling the model.

### Exercise 3: Dockerfile Optimization
The current Dockerfile copies ALL files. Modify it to use a `.dockerignore` file that excludes: `*.pyc`, `__pycache__`, `*.md`, `data/`, `tests/`. How much smaller is the image?

In [ ]:
# Test the running API (run uvicorn first, or test logic directly)
api_test_code = '''
import requests

BASE_URL = "http://localhost:8000"

# Health check
print("=== Health Check ===")
r = requests.get(f"{BASE_URL}/health")
print(r.json())

# Single prediction
print("\\n=== Single Prediction ===")
payload = {
    "tenure_months": 3,
    "monthly_charges": 95.0,
    "total_charges": 285.0,
    "contract_type": "Month-to-month",
    "num_products": 1,
    "support_calls": 3,
}
r = requests.post(f"{BASE_URL}/predict", json=payload)
print(r.json())

# Batch prediction
print("\\n=== Batch Prediction ===")
batch_payload = {"customers": [payload, {**payload, "tenure_months": 24, "monthly_charges": 45}]}
r = requests.post(f"{BASE_URL}/predict/batch", json=batch_payload)
result = r.json()
print(f"Total: {result['total']}, High Risk: {result['high_risk_count']}")
'''

print("API test code (run after starting server):")
print("uvicorn main:app --reload")
print("\nThen run:")
print(api_test_code)

---
## Section 3: Running with Docker

### Build and Run
```bash
# Build the image
docker build -t churn-api:v1 .

# Run the container
docker run -p 8000:8000 churn-api:v1

# Test it
curl http://localhost:8000/health
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"tenure_months": 6, "monthly_charges": 85, "total_charges": 510, "contract_type": "Month-to-month"}'
```

### Docker Commands Cheat Sheet
```bash
docker images                     # List images
docker ps                         # List running containers
docker ps -a                      # List all containers (including stopped)
docker logs <container_id>        # View container logs
docker exec -it <container> bash  # Shell into running container
docker stop <container_id>        # Stop container
docker rm <container_id>          # Delete container
docker rmi <image_id>             # Delete image
```

### Publishing to ECR (AWS Container Registry)
```bash
aws ecr create-repository --repository-name churn-api
aws ecr get-login-password | docker login --username AWS --password-stdin <account>.dkr.ecr.us-east-1.amazonaws.com
docker tag churn-api:v1 <account>.dkr.ecr.us-east-1.amazonaws.com/churn-api:v1
docker push <account>.dkr.ecr.us-east-1.amazonaws.com/churn-api:v1
```

In [ ]:
# Test the API logic without running the server
# In production, use: uvicorn main:app --reload

from pydantic import BaseModel, Field, validator
from typing import Optional

# Simulate the prediction logic
def dummy_predict(tenure_months, monthly_charges, total_charges, contract_type, num_products=1, support_calls=0):
    """Dummy prediction for testing."""
    prob = min(0.95, max(0.05, 0.3 + (monthly_charges - 65) * 0.005 - tenure_months * 0.003))
    risk_tier = "High" if prob > 0.6 else "Medium" if prob > 0.3 else "Low"
    return {
        "churn_probability": round(prob, 4),
        "churn_predicted": prob > 0.5,
        "risk_tier": risk_tier,
    }

# Test cases
test_cases = [
    {"tenure_months": 1, "monthly_charges": 90, "total_charges": 90, "contract_type": "Month-to-month"},
    {"tenure_months": 36, "monthly_charges": 45, "total_charges": 1620, "contract_type": "Two year"},
    {"tenure_months": 12, "monthly_charges": 70, "total_charges": 840, "contract_type": "One year"},
]

print("Prediction Test Results:")
print(f"{'Tenure':>8} {'Monthly':>10} {'Contract':>18} {'Prob':>8} {'Tier':>8}")
print("-" * 60)
for tc in test_cases:
    result = dummy_predict(**tc)
    print(f"{tc['tenure_months']:>8} {tc['monthly_charges']:>10.0f} {tc['contract_type']:>18} {result['churn_probability']:>8.3f} {result['risk_tier']:>8}")

In [ ]:
fastapi_app = '''"""
FastAPI application for customer churn prediction.
"""

import pickle
import time
from typing import List, Optional
import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, validator

# Initialize FastAPI
app = FastAPI(
    title="Churn Prediction API",
    description="Predict customer churn probability",
    version="1.0.0",
)

# Load model at startup (not per-request!)
MODEL = None

@app.on_event("startup")
async def load_model():
    global MODEL
    try:
        with open("model.pkl", "rb") as f:
            MODEL = pickle.load(f)
        print("Model loaded successfully")
    except FileNotFoundError:
        print("WARNING: model.pkl not found — using dummy model")
        MODEL = None


# ---- Request/Response Schemas ----

class CustomerFeatures(BaseModel):
    """Input features for a single customer."""
    tenure_months: int = Field(..., ge=0, le=120, description="Months as customer")
    monthly_charges: float = Field(..., ge=0, le=500, description="Monthly bill amount")
    total_charges: float = Field(..., ge=0, description="Total amount charged")
    contract_type: str = Field(..., description="Month-to-month, One year, Two year")
    num_products: int = Field(default=1, ge=1, le=10)
    support_calls: int = Field(default=0, ge=0, le=20)

    @validator("contract_type")
    def validate_contract(cls, v):
        valid = ["Month-to-month", "One year", "Two year"]
        if v not in valid:
            raise ValueError(f"contract_type must be one of {valid}")
        return v

class PredictionResponse(BaseModel):
    customer_id: Optional[str]
    churn_probability: float
    churn_predicted: bool
    risk_tier: str
    model_version: str = "1.0.0"
    inference_time_ms: float


class BatchRequest(BaseModel):
    customers: List[CustomerFeatures]


# ---- Endpoints ----

@app.get("/health")
async def health_check():
    """Health check — used by load balancers and Docker."""
    return {
        "status": "healthy",
        "model_loaded": MODEL is not None,
    }


@app.get("/")
async def root():
    return {"message": "Churn Prediction API v1.0. See /docs for usage."}


@app.post("/predict", response_model=PredictionResponse)
async def predict(customer: CustomerFeatures, customer_id: Optional[str] = None):
    """Predict churn for a single customer."""
    start = time.time()

    # Prepare features
    features = {
        "tenure_months": customer.tenure_months,
        "monthly_charges": customer.monthly_charges,
        "total_charges": customer.total_charges,
        "num_products": customer.num_products,
        "support_calls": customer.support_calls,
        "contract_encoded": {"Month-to-month": 0, "One year": 1, "Two year": 2}[customer.contract_type],
    }
    X = pd.DataFrame([features])

    if MODEL is not None:
        prob = float(MODEL.predict_proba(X)[0, 1])
    else:
        # Dummy model for demo
        prob = min(0.95, max(0.05, 0.3 + (customer.monthly_charges - 65) * 0.005))

    risk_tier = "High" if prob > 0.6 else "Medium" if prob > 0.3 else "Low"

    return PredictionResponse(
        customer_id=customer_id,
        churn_probability=round(prob, 4),
        churn_predicted=prob > 0.5,
        risk_tier=risk_tier,
        inference_time_ms=round((time.time() - start) * 1000, 2),
    )


@app.post("/predict/batch")
async def predict_batch(request: BatchRequest):
    """Predict churn for multiple customers."""
    if len(request.customers) > 1000:
        raise HTTPException(status_code=400, detail="Maximum 1000 customers per batch")

    results = []
    for i, customer in enumerate(request.customers):
        result = await predict(customer, customer_id=f"batch-{i}")
        results.append(result)

    return {
        "predictions": results,
        "total": len(results),
        "high_risk_count": sum(1 for r in results if r.risk_tier == "High"),
    }


@app.get("/model/info")
async def model_info():
    """Return model metadata."""
    return {
        "version": "1.0.0",
        "algorithm": "GradientBoostingClassifier",
        "features": ["tenure_months", "monthly_charges", "total_charges", "num_products", "support_calls"],
        "trained_on": "2024-01-01",
        "metrics": {"val_auc": 0.82},
    }
'''

print("FastAPI app written")
print("Key endpoints:")
print("  GET  /health     - Health check")
print("  GET  /docs       - Swagger UI (auto-generated!)")
print("  POST /predict    - Single prediction")
print("  POST /predict/batch - Batch predictions")
print("  GET  /model/info - Model metadata")

---
## Section 2: FastAPI for ML Serving

FastAPI is the go-to framework for building ML APIs:
- Auto-generates OpenAPI/Swagger documentation
- Type validation via Pydantic (catches bad inputs before they reach your model)
- Async support for high-throughput serving
- ~100x faster than Flask for concurrent requests

In [ ]:
dockerfile_content = '''# Base image: Python 3.11 slim
FROM python:3.11-slim

# Set working directory inside container
WORKDIR /app

# Install system dependencies
RUN apt-get update && apt-get install -y \\
    gcc \\
    && rm -rf /var/lib/apt/lists/*

# Copy requirements first (for layer caching)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY . .

# Expose port
EXPOSE 8000

# Health check
HEALTHCHECK --interval=30s --timeout=10s --start-period=5s --retries=3 \\
    CMD curl -f http://localhost:8000/health || exit 1

# Run the FastAPI server
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"]
'''

requirements_content = '''fastapi>=0.104.0
uvicorn[standard]>=0.24.0
scikit-learn>=1.3.0
pandas>=2.0.0
numpy>=1.24.0
pydantic>=2.0.0
python-multipart>=0.0.6
'''

print("Dockerfile:")
print(dockerfile_content)
print("\nrequirements.txt:")
print(requirements_content)

---
## Section 1: Why Docker for ML?

### The "Works on My Machine" Problem
- Training: Python 3.11, scikit-learn 1.3.0, numpy 1.24
- Production server: Python 3.8, scikit-learn 0.24, numpy 1.20
- **Result**: Different predictions, crashes, dependency conflicts

### Docker Solves This
A Docker **image** packages:
- Python version
- All dependencies (exact versions)
- Your code
- Config files

Run anywhere: your laptop, EC2, SageMaker, Lambda, Kubernetes.

### Key Docker Concepts
| Concept | Analogy | Description |
|---------|---------|-------------|
| **Image** | Recipe | Blueprint — read-only template |
| **Container** | Cake | Running instance of an image |
| **Dockerfile** | Instructions | How to build the image |
| **Registry** | Recipe book | Store images (Docker Hub, ECR) |
| **docker build** | Bake | Create image from Dockerfile |
| **docker run** | Eat | Start a container from image |

In [ ]:
import json, os, subprocess
import pandas as pd, numpy as np
import warnings; warnings.filterwarnings('ignore')
print("Docker version:", end=" ")
try:
    result = subprocess.run(['docker', '--version'], capture_output=True, text=True)
    print(result.stdout.strip())
except:
    print("Docker not found — install Docker Desktop")

# 03: Docker & FastAPI — Containerized ML Serving

## Learning Objectives
- Understand Docker containers for ML model serving
- Build a FastAPI application to serve model predictions
- Containerize the FastAPI app with Docker
- Test and document your API

## Roadmap
FastAPI basics → Model serving endpoint → Docker container → Testing → Deployment